# SUI Wallet Transfer History

Retrieve all historical SUI transfers involving `0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8` using the public Sui JSON-RPC. The notebook collects balance changes, classifies direction (in/out), lists counterparties, and saves the results for further analysis.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import pandas as pd  # type: ignore
import requests
from datetime import datetime, timezone


In [2]:
RPC_URL = "https://fullnode.mainnet.sui.io:443"
REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
RETRY_BACKOFF = 2.0
MAX_TX_PER_WALLET = 1000
TARGET_ADDRESS = "0xbe90dbe25c747f23e7b93f8948f240a90aa106597b7dab0b33e852b0b1950aa8"
START_DATE = datetime(2024, 5, 1, tzinfo=timezone.utc)  # adjust or set to None


In [3]:
def rpc_call(method: str, params: Optional[Sequence[object]] = None) -> dict:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or []}
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.post(RPC_URL, json=payload, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
            if "error" in data:
                raise RuntimeError(data["error"])
            return data["result"]
        except Exception as exc:
            last_error = exc
            backoff = RETRY_BACKOFF * (attempt + 1)
            print(f"RPC call {method} failed (attempt {attempt+1}/{MAX_RETRIES}); retrying in {backoff:.1f}s…")
            import time
            time.sleep(backoff)
    raise RuntimeError(f"RPC call failed after retries: {method}") from last_error


In [4]:
def fetch_transaction_digests(address: str, filter_key: str) -> List[str]:
    digests: List[str] = []
    cursor = None
    while True:
        result = rpc_call(
            "suix_queryTransactionBlocks",
            [
                {
                    "filter": {filter_key: address},
                    "options": {"showBalanceChanges": True, "showEvents": True},
                },
                cursor,
                100,
                True,
            ],
        )
        digests.extend(row["digest"] for row in result["data"])
        cursor = result.get("nextCursor")
        if not result["hasNextPage"] or len(digests) >= MAX_TX_PER_WALLET:
            break
    return digests


def fetch_transaction_block(digest: str) -> dict:
    return rpc_call(
        "sui_getTransactionBlock",
        [
            digest,
            {
                "showInput": True,
                "showEvents": True,
                "showBalanceChanges": True,
                "showEffects": True,
            },
        ],
    )


In [5]:
def summarise_changes(block: dict, target: str) -> List[dict]:
    s_changes = [c for c in block.get('balanceChanges', []) if c.get('coinType') == '0x2::sui::SUI']
    target_changes = [c for c in s_changes if c.get('owner', {}).get('AddressOwner') == target]
    if not target_changes:
        return []
    counterparty_addrs = sorted({c.get('owner', {}).get('AddressOwner') for c in s_changes if c.get('owner', {}).get('AddressOwner') != target})
    total_mist = sum(int(c.get('amount') or 0) for c in target_changes)
    direction = 'in' if total_mist > 0 else 'out'
    amount_sui = abs(total_mist) / 1_000_000_000
    ts_ms = block.get('timestampMs')
    timestamp = datetime.fromtimestamp(int(ts_ms)/1000, timezone.utc) if ts_ms else None
    return [{
        'tx_digest': block.get('digest'),
        'timestamp': timestamp,
        'direction': direction,
        'amount_sui': amount_sui,
        'counterparties': ';'.join(addr for addr in counterparty_addrs if addr),
    }]


def gather_transfers(address: str, start_date: Optional[datetime] = None) -> pd.DataFrame:
    digests = set(fetch_transaction_digests(address, 'FromAddress') + fetch_transaction_digests(address, 'ToAddress'))
    records: List[dict] = []
    for digest in sorted(digests):
        block = fetch_transaction_block(digest)
        summaries = summarise_changes(block, address)
        for entry in summaries:
            ts = entry['timestamp']
            if start_date and ts and ts < start_date:
                continue
            entry['timestamp'] = ts.isoformat() if ts else ''
            records.append(entry)
    df = pd.DataFrame(records)
    return df.sort_values('timestamp') if not df.empty else df


In [6]:
transfers_df = gather_transfers(TARGET_ADDRESS, START_DATE)
print(f"Total transfer rows: {len(transfers_df)}")
transfers_df.head()


Total transfer rows: 1061


,tx_digest,timestamp,direction,amount_sui,counterparties
603,BLMzbTo6PQdpbDhKgmcm1H9gmYM12mDTG7bPhRD6QnuR,2024-10-14T01:54:23.273000+00:00,out,170877.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
349,7i8oHAG8X5E2ktyAv3b14vMPwr4kbzp3UhBmrpBXoKJH,2024-10-14T02:02:04.051000+00:00,out,113165.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
546,AZk2t1hSVQSMQR4daaAZeD3du3fPpcpEuZBt6H9TDhyi,2024-10-14T02:15:17.515000+00:00,out,149059.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
822,EpnvAeLM225JmWTm6z4d5CwPjZ2K2Tq6DoPsmmZ5Zm2U,2024-10-14T02:53:38.738000+00:00,out,546885.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
520,AA8pZEifAWoweZXkRc6Gn5bKSZwfNSJQHc1e9fs65bwx,2024-10-14T03:03:24.411000+00:00,out,124555.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...


In [7]:
if transfers_df.empty:
    raise RuntimeError('No transfers returned. Increase MAX_TX_PER_WALLET or adjust START_DATE.')

summary = transfers_df.groupby('direction')['amount_sui'].sum()
print('Net SUI by direction:')
print(summary)
transfers_df


Net SUI by direction:
direction
in     1.466549e+08
out    1.602514e+08
Name: amount_sui, dtype: float64


,tx_digest,timestamp,direction,amount_sui,counterparties
603,BLMzbTo6PQdpbDhKgmcm1H9gmYM12mDTG7bPhRD6QnuR,2024-10-14T01:54:23.273000+00:00,out,170877.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
349,7i8oHAG8X5E2ktyAv3b14vMPwr4kbzp3UhBmrpBXoKJH,2024-10-14T02:02:04.051000+00:00,out,113165.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
546,AZk2t1hSVQSMQR4daaAZeD3du3fPpcpEuZBt6H9TDhyi,2024-10-14T02:15:17.515000+00:00,out,149059.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
822,EpnvAeLM225JmWTm6z4d5CwPjZ2K2Tq6DoPsmmZ5Zm2U,2024-10-14T02:53:38.738000+00:00,out,546885.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
520,AA8pZEifAWoweZXkRc6Gn5bKSZwfNSJQHc1e9fs65bwx,2024-10-14T03:03:24.411000+00:00,out,124555.001758,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...
...,...,...,...,...,...
254,6E8saZ3KDvAjQLuR75Bt8QDqb86KknjEhKxDCc5RDzj1,2025-09-17T16:56:26.968000+00:00,in,157000.000000,0x7f3b918653aacd4289b7e471aea588b34d497df0e154...
192,5G8fSs7P2eZ93p9SSd4WbrhDntZWJxGiuEUyVPvxUMcm,2025-09-17T17:32:43.009000+00:00,in,88000.000000,0xa4da59e5ba0dabefd1958fc42b4cbf8f01e165d31154...
367,7yUyApzQjeqsyobRtxnCtTVrqgF8cW9C8y4wf7onx1TU,2025-09-17T18:01:42.789000+00:00,in,78000.000000,0xe79de7eb11df6ee3ba7a37ee97aa7a771762f4a9e307...
21,2DsErexY9iKgmnjA2jJPWuzxgPmZP3Ep6XqrumTgr1cf,2025-09-18T15:53:51.495000+00:00,out,173410.998573,0x457fd3e8295543c23cfc2182b33247338cb04ddf43d0...


In [8]:
daily = transfers_df.copy()
daily['timestamp'] = pd.to_datetime(daily['timestamp'])
daily_totals = daily.groupby([daily['timestamp'].dt.date, 'direction'])['amount_sui'].sum().unstack(fill_value=0)
daily_totals


direction,in,out
timestamp,,
2024-10-14,0.0,6.530478e+06
2024-10-21,0.0,2.962994e+03
2024-10-22,0.0,1.284288e+06
2024-10-23,0.0,4.255790e+05
2024-10-26,0.0,1.024240e+05
...,...,...
2025-08-22,0.0,1.902190e+05
2025-09-01,0.0,1.302110e+05
2025-09-17,323000.0,0.000000e+00


In [9]:
output_dir = Path('/Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/diagnostics')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'wallet_0xbe90db_transfer_history.csv'
transfers_df.to_csv(output_path, index=False)
print(f'Saved transfer history to {output_path}')


Saved transfer history to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/diagnostics/wallet_0xbe90db_transfer_history.csv


## Notes

- Increase `MAX_TX_PER_WALLET` or expand `START_DATE` if you need deeper history (beware RPC rate limits).
- `counterparties` lists the other addresses that had SUI balance changes in the same transaction; you can cross-reference with `known_addresses.csv` for exchange identification.
- Run the daily totals cell to produce a simple time series of inflows/outflows.
